### Import

In [1]:
import os
import sys
import re
import numpy as np
import pandas as pd
import datetime as dt
from matplotlib import pyplot as plt 

pd.set_option('display.max_columns', 500)

### General parameters

In [2]:
path_csv  = "../Data/csvExtract/"
path_timeseries = "../Data/Output"

### Demographics

In [3]:
cohort_df = pd.read_csv(path_csv + "demographic.csv")
cohort_df[['ICUSTAY_ID']] = cohort_df[['ICUSTAY_ID']].astype(int)
cohort_df.head(2)

### Comorbidities

In [4]:
comorbidities = pd.read_csv(path_csv + "comorbidities.csv")
comorbidities = pd.merge(cohort_df, comorbidities, on=['HADM_ID'], how='left')
comorbidities.drop(columns=['GENDER', 'AGE', 'ETHNICITY', 'ADMITTIME', 'DISCHTIME', 'INTIME', 'OUTTIME', 'ICU_LOS_H', 
                            'ICU_LOS_D', 'HOSP_LOS_H', 'HOSP_LOS_D', 'ICU_EXPIRE_FLAG',
                            'HOSPITAL_EXPIRE_FLAG', 'DEATH_TIME_DISCH', 'EXPIRE_FLAG'], inplace=True)
comorbidities = comorbidities.fillna(0)

In [5]:
comorbidities.head(2)

### Feature selection

In [6]:
total_icustay = cohort_df.ICUSTAY_ID.nunique()

In [7]:
def feature_selection(df, all_icustay, threshold):
    
    df1 = df[['ICUSTAY_ID', 'LABEL']]
    df1 = df1.drop_duplicates()
    feature_list = []
    percent_list = []
    
    for feature in df1.LABEL.unique():
        percentage = round((df[df.LABEL == feature].ICUSTAY_ID.nunique())/all_icustay, 2)
        if percentage >= threshold:
            feature_list.append(feature)
            percent_list.append(percentage)
        
    columns = {'variable':feature_list,'percent':percent_list}
    frame = pd.DataFrame(columns)
    frame = frame.sort_values(by='percent', ascending=False).reset_index(drop=True)
    list_features = list(frame.variable.unique())
    
    return frame, list_features

### Lab event

In [8]:
lab_df = pd.read_csv(path_csv  + "lab_event.csv")
lab_df.head(2)

In [9]:
lab_var = [
    
'Temperature',
'PT',
'PTT',
'INR(PT)',
'pH',    
'Lactate',
'Lactate Dehydrogenase (LD)', 'Lactate Dehydrogenase, Pleural',
'Base Excess', 
'Anion Gap',
'Bicarbonate', 'Calculated Bicarbonate, Whole Blood',
'Creatinine',
'Hematocrit', 'Hematocrit, Calculated',
'Hemoglobin', 'Absolute Hemoglobin',
'Bilirubin, Total', 
'Bilirubin, Direct', 
'Bilirubin, Indirect',   
'Urea Nitrogen', 
'MCV',
'MCH',
'MCHC',
'RDW',
'RBC',
'WBC',
'Red Blood Cells', 
'White Blood Cells', 'WBC Count', 
'Platelet Count',
'Glucose', 
'Ammonia',
'Magnesium', 
'Phosphate', 
'Alkaline Phosphatase',
'Potassium', 'Potassium, Whole Blood', 
'Sodium', 'Sodium, Whole Blood',  
'Chloride', 'Chloride, Whole Blood', 
'Free Calcium', 
'Calcium, Total',  
'Cholesterol, Total', 
'Cholesterol, HDL', 
'Cholesterol, LDL, Calculated', 'Cholesterol, LDL, Measured',
'C-Reactive Protein',
'pO2', 
'pCO2',  
'Alanine Aminotransferase (ALT)',
'Asparate Aminotransferase (AST)',
'Bands',
'Polys',
'Amylase', 'Amylase, Ascites', 'Amylase, Pleural',
'Lipase', 
'Lymphocytes', 
'Monocytes',
'Eosinophils', 
'Basophils', 
'Neutrophils',
'Absolute Lymphocyte Count', 
'Monocyte Count', 
'Eosinophil Count',
'O2 Flow',
'Oxygen', 
'Oxygen Saturation',
'Calculated Total CO2',
'Albumin', 'Albumin, Ascites',  'Albumin, Pleural', 
'Troponin T', 
'Troponin I',
'Vancomycin',
'Triglycerides', 'Triglycerides, Pleural', 'Triglycerides, Ascites', 
'Fibrinogen, Functional', 
'Transferrin',
'Ferritin',
'Cortisol', 
'Protein',
'Protein, Total', 'Total Protein, Pleural', 'Total Protein, Ascites', 
'PEEP',
'Tidal Volume',    
'Intubated',
'Ventilator',
'Ventilation Rate',
'Granulocyte Count',
'Promyelocytes',
'Metamyelocytes',
'Myelocytes',
'Ovalocytes',
]

In [10]:
lab_df = lab_df[lab_df.LABEL.isin(lab_var)]
lab_df = lab_df[lab_df.VALUE.notnull()]

In [11]:
def non_float_items(lst):
    result = []
    for item in lst:
        try:
            float(item)
        except ValueError:
            result.append(item)
    return result

In [ ]:
for var in lab_var:
    if var not in ['Ventilator']:
        try:
            sub_df = lab_df[lab_df.LABEL == var]
            values = sub_df.VALUE.astype(float)
            
        except ValueError:
            sub_df = lab_df[lab_df.LABEL == var]
            uniq_val = sub_df.VALUE.unique()
            non_float_val = non_float_items(uniq_val)
            lab_df.loc[(lab_df['LABEL'] == var)  & (lab_df['VALUE'].isin(non_float_val)), 'VALUE'] = np.nan
            print(var)
            print("Faild")
            print("************")
            
lab_df = lab_df[lab_df.VALUE.notnull()]

In [13]:
lab_df.loc[lab_df['LABEL'] == 'O2 Flow' , 'VALUE'] = (lab_df[lab_df['LABEL'] == 'O2 Flow'].VALUE.astype(float) / 10) 

In [14]:
lab_df.loc[lab_df['LABEL'] == 'Lactate Dehydrogenase, Pleural', 'LABEL'] = 'Lactate Dehydrogenase (LD)'

lab_df.loc[lab_df['LABEL'] == 'Calculated Bicarbonate, Whole Blood', 'LABEL'] = 'Bicarbonate'

lab_df.loc[lab_df['LABEL'] == 'Hematocrit, Calculated', 'LABEL'] = 'Hematocrit'
lab_df.loc[lab_df['LABEL'] == 'Absolute Hemoglobin', 'LABEL'] = 'Hemoglobin'
lab_df.loc[lab_df['LABEL'] == 'WBC Count', 'LABEL'] = 'White Blood Cells'

lab_df.loc[lab_df['LABEL'] == 'Potassium, Whole Blood', 'LABEL'] = 'Potassium'
lab_df.loc[lab_df['LABEL'] == 'Sodium, Whole Blood', 'LABEL'] = 'Sodium'
lab_df.loc[lab_df['LABEL'] == 'Chloride, Whole Blood', 'LABEL'] = 'Chloride'
lab_df.loc[lab_df['LABEL'] == 'Cholesterol, LDL, Calculated', 'LABEL'] = 'Cholesterol, LDL'
lab_df.loc[lab_df['LABEL'] == 'Cholesterol, LDL, Measured',   'LABEL'] = 'Cholesterol, LDL'
lab_df.loc[lab_df['LABEL'] == 'Free Calcium', 'LABEL'] = 'Ionized Calcium'

lab_df.loc[lab_df['LABEL'] == 'C-Reactive Protein', 'LABEL'] = 'C-Reactive Protein (CRP)'

lab_df.loc[lab_df['LABEL'] == 'Amylase, Ascites', 'LABEL'] = 'Amylase'
lab_df.loc[lab_df['LABEL'] == 'Amylase, Pleural', 'LABEL'] = 'Amylase'

lab_df.loc[lab_df['LABEL'] == 'Albumin, Ascites', 'LABEL'] = 'Albumin'
lab_df.loc[lab_df['LABEL'] == 'Albumin, Pleural', 'LABEL'] = 'Albumin'

lab_df.loc[lab_df['LABEL'] == 'Triglycerides, Pleural', 'LABEL'] = 'Triglycerides'
lab_df.loc[lab_df['LABEL'] == 'Triglycerides, Ascites', 'LABEL'] = 'Triglycerides'

lab_df.loc[lab_df['LABEL'] == 'Protein, Total',         'LABEL'] = 'Total Protein'
lab_df.loc[lab_df['LABEL'] == 'Total Protein, Pleural', 'LABEL'] = 'Total Protein'
lab_df.loc[lab_df['LABEL'] == 'Total Protein, Ascites', 'LABEL'] = 'Total Protein'

lab_df.loc[lab_df['LABEL'] == 'Alanine Aminotransferase (ALT)',  'LABEL'] = 'ALT'
lab_df.loc[lab_df['LABEL'] == 'Asparate Aminotransferase (AST)', 'LABEL'] = 'AST'

lab_df.loc[lab_df['LABEL'] == 'Calculated Total CO2', 'LABEL'] = 'Total CO2'
lab_df.loc[lab_df['LABEL'] == 'Fibrinogen, Functional', 'LABEL'] = 'Fibrinogen'
lab_df.loc[lab_df['LABEL'] == 'Urea Nitrogen', 'LABEL'] = 'BUN'

lab_df.loc[lab_df['LABEL'] == 'Neutrophils', 'LABEL'] = 'Differential-Neuts'
lab_df.loc[lab_df['LABEL'] == 'Lymphocytes', 'LABEL'] = 'Differential-Lymphs'
lab_df.loc[lab_df['LABEL'] == 'Basophils',   'LABEL'] = 'Differential-Basos'
lab_df.loc[lab_df['LABEL'] == 'Monocytes',   'LABEL'] = 'Differential-Monos'
lab_df.loc[lab_df['LABEL'] == 'Eosinophils', 'LABEL'] = 'Differential-Eos'
lab_df.loc[lab_df['LABEL'] == 'Bands',       'LABEL'] = 'Differential-Bands'
lab_df.loc[lab_df['LABEL'] == 'Polys',       'LABEL'] = 'Differential-Polys'

In [15]:
lab_df.head(2)

### Chart event

In [16]:
chart_df = pd.read_csv(path_csv  + "chart_event.csv")
chart_df.head(2)

<ipython-input>:1: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  chart_df = pd.read_csv(path_csv  + "chart_event.csv")


In [17]:
chart_var = [
    
'Heart Rhythm',
'Heart Rate',
'Temperature F', 'Temperature Fahrenheit', 'Temperature F (calc)',
'Temperature C', 'Temperature Celsius', 'Temperature C (calc)',
'Skin Temperature', 'Skin [Temperature]',
'Respiratory Rate',  'Resp. Rate', 'Resp Rate',
'Respiratory Rate (spontaneous)', 'Spon RR (Mech.)', 'Spont RR', 'Resp Rate (Spont)',  'Spont Resp rate',
'Respiratory Rate (Total)', 'Resp Rate (Total)',
'Respiratory Rate (Set)', 'Respiratory Rate Set',
'Non Invasive Blood Pressure mean', 'NBP Mean',  
'Non Invasive Blood Pressure diastolic', 'NBP [Diastolic]', 
'Non Invasive Blood Pressure systolic' , 'NBP [Systolic]', 
'Arterial Blood Pressure mean', 'Arterial BP Mean', 'ART BP mean', 'Arterial BP Mean #2', 
'Arterial Blood Pressure diastolic', 'Arterial BP [Diastolic]', 'ART BP Diastolic', 'Arterial BP #2 [Diastolic]',  
'Arterial Blood Pressure systolic',  'Arterial BP [Systolic]', 'ART BP Systolic', 'Arterial BP #2 [Systolic]', 
'Pulmonary Artery Pressure mean',      'PAP Mean',        'PA mean pressure (PA Line)',
'Pulmonary Artery Pressure diastolic', 'PAP [Diastolic]', 'PA diastolic pressure(PA Line)',
'Pulmonary Artery Pressure systolic',  'PAP [Systolic]',  'PA systolic pressure(PA Line)',
'Arterial O2 pressure',  'Arterial PaO2', 'Venous O2 Pressure',
'Arterial CO2 Pressure', 'Arterial PaCO2', 'PCO2', 'pCO2', 'Venous CO2 Pressure',  
'Mean Airway Pressure', 'MEAN AIRWAY PRESS',
'Pain Level', 'Pain Level (Movemnt)', 'Pain Level (Rest)', 'Pain Level/Response', 'Pain Level Response',
'Pain', 'Pain (0-10)',
'Pain Present',
'GCS Total',
'GCS - Eye Opening'    , 'Eye Opening',    
'GCS - Motor Response' , 'Motor Response', 
'GCS - Verbal Response', 'Verbal Response', 
'Mental status',    
'Richmond-RAS Scale',   
'Goal Richmond-RAS Scale',
'Risk for Falls',
'Delirium assessment',
'CAM-ICU MS Change', 'CAM-ICU MS change',   
'CAM-ICU RASS LOC',
'CAM-ICU Inattention',   
'CAM-ICU Altered LOC',
'CAM-ICU Disorganized thinking',    
'Sedation Score',
'Ramsey SedationScale',
'Flow Rate (L/min)',
'O2 Flow', 'O2 Flow (lpm)', 'O2 Flow (lpm) #2', 'O2 Flow (additional cannula)',
'Inspired O2 Fraction', 'FIO2', 'FiO2', 'FIO2 [Meas]', 'FiO2 (Analyzed)', 
'FIO2 SET', 'FiO2 Set',
'O2 saturation pulseoxymetry', 'SpO2',   
'Arterial O2 Saturation', 'SaO2', 'SaO2 (post)',
'SvO2', 'svo2', 'SVO2','lab svo2', 'SVO2 SAT', 'Central line SVO2',   
'Admission Weight (Kg)', 'Weight Kg', 'Present Weight  (kg)',
'Admission Weight (lbs.)', 'Present Weight  (lb)',
'Height (cm)', 'Admit Ht', 'Height Inches', 
'Length Calc Inches', 'Length in Inches', 'Length Calc (cm)', 'Length in cm',   
'Glucose', 'Glucose (serum)', 'Blood Glucose', 'BloodGlucose', 'Glucose Monitor #', 'abg: glucose', 'Glucose (70-105)',
'Fingerstick Glucose', 'fingerstick glucose', 'FINGERSTICK GLUCOSE.', 'finger stick glucose', 'Glucose finger stick',
'Glucose (whole blood)', 
'cvp', 'CVP', 'Central Venous Pressure', 
'ETCO2', 'EtCO2', 
'EtCO2 Clinical indication',    
'ALT',
'AST', 
'INR', 'INR (2-4 ref. range)',
'PT', 'PT(11-13.5)',  'Prothrombin time', 
'PTT', 'Ptt', 'PTT(22-35)',
'WBC', 'WBC   (4-11,000)', 'WBC (4-11,000)', 'WBC 4.0-11.0',
'Platelets', 'Platelet  (150-440)', 'Platelet Count',
'Magnesium','Magnesium (1.6-2.6)', 
'Phosphorous', 'Phosphorous(2.7-4.5)',   
'Sodium', 'Sodium (serum)',  'Sodium (135-148)', 'Sodium  (135-148)', 'ABG Sodium', 'ABG SODIUM', 'Sodium (whole blood)',
'Chloride', 'Chloride (serum)',  'Chloride (100-112)', 'Chloride  (100-112)', 'ABG Chloride', 'Chloride (whole blood)', 
'Potassium', 'Potassium (serum)', 'Potassium (3.5-5.3)', 'Potassium  (3.5-5.3)', 'ABG Potassium', 'ABG POTASSIUM', 'Potassium (whole blood)',
'Ionized Calcium', 'Ionized calcium', 'ionized calcium', 'IONIZED CALCIUM',
'Calcium non-ionized', 
'Alkaline Phosphatase', 'Alkaline Phosphate', 'Alk. Phosphate',
'BUN', 'bun', 'BUN (6-20)', 'BUN    (6-20)',
'Anion gap', 'Anion Gap   (8-20)',
'Lactic Acid', 'Lactic Acid(0.5-2.0)', 
'HCO3', 'HCO3 (serum)', 
'Creatinine', 'Creatinine (0-1.3)', 'Creatinine   (0-0.7)', 
'Hemoglobin',
'Hematocrit', 'Hematocrit (serum)', 'Hematocrit (35-51)',
'Bilirubin', 'Direct Bilirubin', 'Direct Bili', 'Direct Bili (0-0.3)',
'Indirect Bili(0-1.0)', 
'Total Bilirubin', 'Total Bili (0-1.5)', 'Total Bili', 
'Base Excess', 'Arterial Base Excess', 'Venous Base Excess', 'Base Excess (cap)',
'PH', 'Ph', 'ph level', 'PH (Arterial)', 'pH (Art)', 'Arterial pH', 'Art.pH', 'PH (Venous)', 'Venous pH', 'pH (cap)', 
'LDH', 
'Fibrinogen', 'FIBRINOGEN', 'Fibrinogen (150-400)', 
'Troponin-T', 
'Troponin',  
'Plateau Pressure',   
'Triglyceride', 'Triglyceride (0-200)', 'Triglyceride (0-250)',  
'Vancomycin',
'Vancomycin/Trough', 'Vancomycin (Trough)', 
'Vancomycin/Peak', 'Vancomycin (Peak)',
'Vancomycin/Random', 'Vancomycin (Random)', 
'PEEP Set', 'PEEP set',  
'PEEP', 'MEASURED PEEP', 'Intrinsic peep', 'Total PEEP Level', 'total PeeP',
'Auto-PEEP level', 'Auto-PEEP Level', 'autopeep', 'Autopeep', 'AUTOPeeP', 'auto-peep', 'Auto PeeP', 
'Tidal Volume (Set)', 'Tidal Volume (set)',
'Tidal Volume', 'tidal volumes', 'TIDAL VOLUME', 'tidal volume', 'tidal vol', 
'Tidal Volume (Obser)', 'Tidal Volume (observed)', 'Tidal Volume (spontaneous)', 'spont tidal volumes',
'Spont. Tidal Volume', 'Tidal Volume (Spont)', 'spont Tidal volumes',  'SPNIOT TIDAL VOLUMES', 'sp tidal volumes',
'Ventilator Mode', 
'Ventilator Type', 
'Differential-Eos', 
'Differential-Basos', 
'Differential-Lymphs', 
'Differential-Neuts',
'Differential-Monos',
'Differential-Bands',
'Differential-Polys', 
'Amylase',
'Lipase', 'lipase',  
'Total Protein', 'Total Protein(6.5-8)', 'T. Protein (5-7.5)', 
'C Reactive Protein (CRP)', 
'Ammonia', 'ammonia', 'AMMONIA', 'AMMONIA/12-47 UMOL/L',   
'RBC', 'RBC(3.6-6.2)', 'PRBCS', 'PRBCs', 'PRBC',   
'Cortisol', 'cortisol', 'CORTISOL LEVEL PRE',   
'Albumin', 'ALBUMIN', 'albumin', 'Albumin (>3.2)', 'Albumin  (3.9-4.8)', 
'Arterial CO2(Calc)', 'CaO2', 'Venous CO2', 'Carbon Dioxide', 'Venous CO2(Calc)',
'TCO2 (calc) Arterial', 'TCO2 (calc) Venous', 'Total CO2', 'TCO2 (cap)', 
'TcCO2 [Value]', 'TcO2 [Value]', 'TCO2 (calc) Venous', 'Venous TCO2',  'TCO2        (21-30)', 
'Pressure Support', 'pressure support', 'PRESSURE SUPPORT',   
]

In [18]:
chart_df = chart_df[chart_df.LABEL.isin(chart_var)]
chart_df = chart_df[chart_df.VALUE.notnull()]

In [19]:
def non_float_items(lst):
    result = []
    for item in lst:
        try:
            float(item)
        except ValueError:
            result.append(item)
    return result

In [ ]:
for var in chart_var:
    if var not in ['Heart Rhythm', 'Ventilator Mode', 'Ventilator Type', 'Sedation Score', 'Ramsey SedationScale',
                   'EtCO2 Clinical indication']:          
        try:
            sub_df = chart_df[chart_df.LABEL == var]
            values = sub_df.VALUE.astype(float)
            
        except ValueError:
            sub_df = chart_df[chart_df.LABEL == var]
            uniq_val = sub_df.VALUE.unique()
            non_float_val = non_float_items(uniq_val)
            chart_df.loc[(chart_df['LABEL'] == var)  & (chart_df['VALUE'].isin(non_float_val)), 'VALUE'] = np.nan
            print(var)
            print("Faild")
            print("************")
            
chart_df = chart_df[chart_df.VALUE.notnull()]

In [21]:
chart_df.loc[chart_df['LABEL'] == 'Temperature F' , 'VALUE'] = (chart_df[chart_df['LABEL'] == 'Temperature F'].VALUE.astype(float) - 32) * 5 / 9
chart_df.loc[chart_df['LABEL'] == 'Temperature Fahrenheit' , 'VALUE'] = (chart_df[chart_df['LABEL'] == 'Temperature Fahrenheit'].VALUE.astype(float) - 32) * 5 / 9
chart_df.loc[chart_df['LABEL'] == 'Temperature F (calc)' , 'VALUE'] = (chart_df[chart_df['LABEL'] == 'Temperature F (calc)'].VALUE.astype(float) - 32) * 5 / 9

chart_df.loc[chart_df['LABEL'] == 'FiO2 (Analyzed)' , 'VALUE'] = (chart_df[chart_df['LABEL'] == 'FiO2 (Analyzed)'].VALUE.astype(float) * 100) 
chart_df.loc[chart_df['LABEL'] == 'FiO2 Set' , 'VALUE'] = (chart_df[chart_df['LABEL'] == 'FiO2 Set'].VALUE.astype(float) * 100) 

chart_df.loc[chart_df['LABEL'] == 'Admission Weight (lbs.)' , 'VALUE'] = (chart_df[chart_df['LABEL'] == 'Admission Weight (lbs.)'].VALUE.astype(float) * 0.45)
chart_df.loc[chart_df['LABEL'] == 'Present Weight  (lb)' , 'VALUE'] = (chart_df[chart_df['LABEL'] == 'Present Weight  (lb)'].VALUE.astype(float) * 0.45)

chart_df.loc[chart_df['LABEL'] == 'Admit Ht' , 'VALUE'] = (chart_df[chart_df['LABEL'] == 'Admit Ht'].VALUE.astype(float) * 2.54)
chart_df.loc[chart_df['LABEL'] == 'Height Inches' , 'VALUE'] = (chart_df[chart_df['LABEL'] == 'Height Inches'].VALUE.astype(float) * 2.54)
chart_df.loc[chart_df['LABEL'] == 'Length Calc Inches' , 'VALUE'] = (chart_df[chart_df['LABEL'] == 'Length Calc Inches'].VALUE.astype(float) * 2.54)
chart_df.loc[chart_df['LABEL'] == 'Length in Inches' , 'VALUE'] = (chart_df[chart_df['LABEL'] == 'Length in Inches'].VALUE.astype(float) * 2.54)

In [22]:
chart_df.loc[chart_df['LABEL'] == 'Temperature F',           'LABEL'] = 'Temperature'
chart_df.loc[chart_df['LABEL'] == 'Temperature Fahrenheit',  'LABEL'] = 'Temperature'
chart_df.loc[chart_df['LABEL'] == 'Temperature F (calc)',    'LABEL'] = 'Temperature'
chart_df.loc[chart_df['LABEL'] == 'Temperature C',           'LABEL'] = 'Temperature'
chart_df.loc[chart_df['LABEL'] == 'Temperature Celsius',     'LABEL'] = 'Temperature'
chart_df.loc[chart_df['LABEL'] == 'Temperature C (calc)',    'LABEL'] = 'Temperature'

chart_df.loc[chart_df['LABEL'] == 'Skin [Temperature])', 'LABEL'] = 'Skin Temperature'

chart_df.loc[chart_df['LABEL'] == 'Resp. Rate',        'LABEL'] = 'Respiratory Rate'
chart_df.loc[chart_df['LABEL'] == 'Resp Rate',         'LABEL'] = 'Respiratory Rate'
chart_df.loc[chart_df['LABEL'] == 'Spon RR (Mech.)',   'LABEL'] = 'Respiratory Rate'
chart_df.loc[chart_df['LABEL'] == 'Spont RR',          'LABEL'] = 'Respiratory Rate'
chart_df.loc[chart_df['LABEL'] == 'Resp Rate (Spont)', 'LABEL'] = 'Respiratory Rate'
chart_df.loc[chart_df['LABEL'] == 'Spont Resp rate',   'LABEL'] = 'Respiratory Rate'
chart_df.loc[chart_df['LABEL'] == 'Respiratory Rate (spontaneous)', 'LABEL'] = 'Respiratory Rate'

chart_df.loc[chart_df['LABEL'] == 'Resp Rate (Total)',    'LABEL'] = 'Respiratory Rate (Total)'
chart_df.loc[chart_df['LABEL'] == 'Respiratory Rate Set', 'LABEL'] = 'Respiratory Rate (Set)'

chart_df.loc[chart_df['LABEL'] == 'NBP Mean',        'LABEL'] = 'Non Invasive Blood Pressure mean'
chart_df.loc[chart_df['LABEL'] == 'NBP [Diastolic]', 'LABEL'] = 'Non Invasive Blood Pressure diastolic'
chart_df.loc[chart_df['LABEL'] == 'NBP [Systolic]',  'LABEL'] = 'Non Invasive Blood Pressure systolic'

chart_df.loc[chart_df['LABEL'] == 'Arterial BP Mean',           'LABEL'] = 'Arterial Blood Pressure mean'
chart_df.loc[chart_df['LABEL'] == 'ART BP mean',                'LABEL'] = 'Arterial Blood Pressure mean'
chart_df.loc[chart_df['LABEL'] == 'Arterial BP Mean #2',        'LABEL'] = 'Arterial Blood Pressure mean'
chart_df.loc[chart_df['LABEL'] == 'Arterial BP [Diastolic]',    'LABEL'] = 'Arterial Blood Pressure diastolic'
chart_df.loc[chart_df['LABEL'] == 'ART BP Diastolic',           'LABEL'] = 'Arterial Blood Pressure diastolic'
chart_df.loc[chart_df['LABEL'] == 'Arterial BP #2 [Diastolic]', 'LABEL'] = 'Arterial Blood Pressure diastolic'
chart_df.loc[chart_df['LABEL'] == 'Arterial BP [Systolic]',     'LABEL'] = 'Arterial Blood Pressure systolic'
chart_df.loc[chart_df['LABEL'] == 'ART BP Systolic',            'LABEL'] = 'Arterial Blood Pressure systolic'
chart_df.loc[chart_df['LABEL'] == 'Arterial BP #2 [Systolic]',  'LABEL'] = 'Arterial Blood Pressure systolic'

chart_df.loc[chart_df['LABEL'] == 'PAP Mean',                       'LABEL'] = 'Pulmonary Artery Pressure mean'
chart_df.loc[chart_df['LABEL'] == 'PA mean pressure (PA Line)',     'LABEL'] = 'Pulmonary Artery Pressure mean'
chart_df.loc[chart_df['LABEL'] == 'PAP [Diastolic]',                'LABEL'] = 'Pulmonary Artery Pressure diastolic'
chart_df.loc[chart_df['LABEL'] == 'PA diastolic pressure(PA Line)', 'LABEL'] = 'Pulmonary Artery Pressure diastolic'
chart_df.loc[chart_df['LABEL'] == 'PAP [Systolic]',                 'LABEL'] = 'Pulmonary Artery Pressure systolic'
chart_df.loc[chart_df['LABEL'] == 'PA systolic pressure(PA Line)',  'LABEL'] = 'Pulmonary Artery Pressure systolic'

chart_df.loc[chart_df['LABEL'] == 'Arterial O2 pressure',   'LABEL'] = 'pO2'
chart_df.loc[chart_df['LABEL'] == 'Arterial PaO2',          'LABEL'] = 'pO2'
chart_df.loc[chart_df['LABEL'] == 'Venous O2 Pressure',     'LABEL'] = 'pO2'
chart_df.loc[chart_df['LABEL'] == 'Arterial CO2 Pressure',  'LABEL'] = 'pCO2'
chart_df.loc[chart_df['LABEL'] == 'Arterial PaCO2',         'LABEL'] = 'pCO2'
chart_df.loc[chart_df['LABEL'] == 'PCO2',                   'LABEL'] = 'pCO2'
chart_df.loc[chart_df['LABEL'] == 'pCO2',                   'LABEL'] = 'pCO2'
chart_df.loc[chart_df['LABEL'] == 'Venous CO2 Pressure',    'LABEL'] = 'pCO2'

chart_df.loc[chart_df['LABEL'] == 'MEAN AIRWAY PRESS', 'LABEL'] = 'Mean Airway Pressure'

chart_df.loc[chart_df['LABEL'] == 'Pain Level (Movemnt)', 'LABEL'] = 'Pain Level'
chart_df.loc[chart_df['LABEL'] == 'Pain Level (Rest)',    'LABEL'] = 'Pain Level'
chart_df.loc[chart_df['LABEL'] == 'Pain Level/Response',  'LABEL'] = 'Pain Level'
chart_df.loc[chart_df['LABEL'] == 'Pain Level Response',  'LABEL'] = 'Pain Level'
chart_df.loc[chart_df['LABEL'] == 'Pain',                 'LABEL'] = 'Pain Level'
chart_df.loc[chart_df['LABEL'] == 'Pain (0-10)',          'LABEL'] = 'Pain Level'

chart_df.loc[chart_df['LABEL'] == 'Eye Opening',     'LABEL'] = 'GCS - Eye Opening'
chart_df.loc[chart_df['LABEL'] == 'Motor Response',  'LABEL'] = 'GCS - Motor Response'
chart_df.loc[chart_df['LABEL'] == 'Verbal Response', 'LABEL'] = 'GCS - Verbal Response'

chart_df.loc[chart_df['LABEL'] == 'CAM-ICU MS change', 'LABEL'] = 'CAM-ICU MS Change'

chart_df.loc[chart_df['LABEL'] == 'Ramsey SedationScale', 'LABEL'] = 'Sedation Scale'

chart_df.loc[chart_df['LABEL'] == 'O2 Flow (lpm)',                'LABEL'] = 'O2 Flow'
chart_df.loc[chart_df['LABEL'] == 'O2 Flow (lpm) #2',             'LABEL'] = 'O2 Flow'
chart_df.loc[chart_df['LABEL'] == 'O2 Flow (additional cannula)', 'LABEL'] = 'O2 Flow'

chart_df.loc[chart_df['LABEL'] == 'Inspired O2 Fraction', 'LABEL'] = 'FiO2'
chart_df.loc[chart_df['LABEL'] == 'FIO2',                 'LABEL'] = 'FiO2'
chart_df.loc[chart_df['LABEL'] == 'FIO2 [Meas]',          'LABEL'] = 'FiO2'
chart_df.loc[chart_df['LABEL'] == 'FiO2 (Analyzed)',      'LABEL'] = 'FiO2'
chart_df.loc[chart_df['LABEL'] == 'FIO2 SET', 'LABEL'] = 'FiO2 (Set)'
chart_df.loc[chart_df['LABEL'] == 'FiO2 Set', 'LABEL'] = 'FiO2 (Set)'

chart_df.loc[chart_df['LABEL'] == 'O2 saturation pulseoxymetry', 'LABEL'] = 'SpO2'

chart_df.loc[chart_df['LABEL'] == 'Arterial O2 Saturation', 'LABEL'] = 'Oxygen Saturation'
chart_df.loc[chart_df['LABEL'] == 'SaO2 (post)',            'LABEL'] = 'Oxygen Saturation'
chart_df.loc[chart_df['LABEL'] == 'SaO2',                   'LABEL'] = 'Oxygen Saturation'

chart_df.loc[chart_df['LABEL'] == 'svo2',     'LABEL'] = 'SvO2'
chart_df.loc[chart_df['LABEL'] == 'SVO2',     'LABEL'] = 'SvO2'
chart_df.loc[chart_df['LABEL'] == 'lab svo2', 'LABEL'] = 'SvO2'
chart_df.loc[chart_df['LABEL'] == 'SVO2 SAT', 'LABEL'] = 'SvO2'
chart_df.loc[chart_df['LABEL'] == 'Central line SVO2', 'LABEL'] = 'SvO2'

chart_df.loc[chart_df['LABEL'] == 'Weight Kg',               'LABEL'] = 'Admission Weight (Kg)'
chart_df.loc[chart_df['LABEL'] == 'Present Weight  (kg)',    'LABEL'] = 'Admission Weight (Kg)'
chart_df.loc[chart_df['LABEL'] == 'Admission Weight (lbs.)', 'LABEL'] = 'Admission Weight (Kg)'
chart_df.loc[chart_df['LABEL'] == 'Present Weight  (lb)',    'LABEL'] = 'Admission Weight (Kg)'

chart_df.loc[chart_df['LABEL'] == 'Admit Ht',            'LABEL'] = 'Height (cm)'
chart_df.loc[chart_df['LABEL'] == 'Height Inches',       'LABEL'] = 'Height (cm)'
chart_df.loc[chart_df['LABEL'] == 'Length Calc Inches',  'LABEL'] = 'Height (cm)'
chart_df.loc[chart_df['LABEL'] == 'Length in Inches',    'LABEL'] = 'Height (cm)'
chart_df.loc[chart_df['LABEL'] == 'Length Calc (cm)',    'LABEL'] = 'Height (cm)'
chart_df.loc[chart_df['LABEL'] == 'Length in cm',        'LABEL'] = 'Height (cm)'

chart_df.loc[chart_df['LABEL'] == 'Glucose (serum)',        'LABEL'] = 'Glucose'
chart_df.loc[chart_df['LABEL'] == 'Blood Glucose',          'LABEL'] = 'Glucose'
chart_df.loc[chart_df['LABEL'] == 'BloodGlucose',           'LABEL'] = 'Glucose'
chart_df.loc[chart_df['LABEL'] == 'abg: glucose',           'LABEL'] = 'Glucose'
chart_df.loc[chart_df['LABEL'] == 'Glucose (70-105)',       'LABEL'] = 'Glucose' 
chart_df.loc[chart_df['LABEL'] == 'Glucose (whole blood)',  'LABEL'] = 'Glucose'
chart_df.loc[chart_df['LABEL'] == 'Fingerstick Glucose',    'LABEL'] = 'Glucose'
chart_df.loc[chart_df['LABEL'] == 'fingerstick glucose',    'LABEL'] = 'Glucose'
chart_df.loc[chart_df['LABEL'] == 'FINGERSTICK GLUCOSE.',   'LABEL'] = 'Glucose'
chart_df.loc[chart_df['LABEL'] == 'finger stick glucose',   'LABEL'] = 'Glucose'
chart_df.loc[chart_df['LABEL'] == 'Glucose finger stick',   'LABEL'] = 'Glucose'
chart_df.loc[chart_df['LABEL'] == 'Glucose Monitor #',      'LABEL'] = 'Glucose'

chart_df.loc[chart_df['LABEL'] == 'cvp',                     'LABEL'] = 'CVP'
chart_df.loc[chart_df['LABEL'] == 'Central Venous Pressure', 'LABEL'] = 'CVP'

chart_df.loc[chart_df['LABEL'] == 'EtCO2',       'LABEL'] = 'ETCO2'

chart_df.loc[chart_df['LABEL'] == 'INR',                  'LABEL'] = 'INR(PT)'
chart_df.loc[chart_df['LABEL'] == 'INR (2-4 ref. range)', 'LABEL'] = 'INR(PT)'
chart_df.loc[chart_df['LABEL'] == 'PT(11-13.5)',      'LABEL'] = 'PT'
chart_df.loc[chart_df['LABEL'] == 'Prothrombin time', 'LABEL'] = 'PT'
chart_df.loc[chart_df['LABEL'] == 'Ptt',              'LABEL'] = 'PTT'
chart_df.loc[chart_df['LABEL'] == 'PTT(22-35)',       'LABEL'] = 'PTT'

chart_df.loc[chart_df['LABEL'] == 'WBC   (4-11,000)', 'LABEL'] = 'WBC'
chart_df.loc[chart_df['LABEL'] == 'WBC (4-11,000)',   'LABEL'] = 'WBC'
chart_df.loc[chart_df['LABEL'] == 'WBC 4.0-11.0',     'LABEL'] = 'WBC'

chart_df.loc[chart_df['LABEL'] == 'Platelets',           'LABEL'] = 'Platelet Count'
chart_df.loc[chart_df['LABEL'] == 'Platelet  (150-440)', 'LABEL'] = 'Platelet Count'
chart_df.loc[chart_df['LABEL'] == 'Platelet Count',      'LABEL'] = 'Platelet Count'

chart_df.loc[chart_df['LABEL'] == 'Magnesium (1.6-2.6)',  'LABEL'] = 'Magnesium' 
chart_df.loc[chart_df['LABEL'] == 'Phosphorous',          'LABEL'] = 'Phosphate'
chart_df.loc[chart_df['LABEL'] == 'Phosphorous(2.7-4.5)', 'LABEL'] = 'Phosphate'

chart_df.loc[chart_df['LABEL'] == 'Sodium (serum)',       'LABEL'] = 'Sodium'
chart_df.loc[chart_df['LABEL'] == 'Sodium (135-148)',     'LABEL'] = 'Sodium'
chart_df.loc[chart_df['LABEL'] == 'Sodium  (135-148)',    'LABEL'] = 'Sodium'
chart_df.loc[chart_df['LABEL'] == 'ABG Sodium',           'LABEL'] = 'Sodium'
chart_df.loc[chart_df['LABEL'] == 'ABG SODIUM',           'LABEL'] = 'Sodium'
chart_df.loc[chart_df['LABEL'] == 'Sodium (whole blood)', 'LABEL'] = 'Sodium'
 
chart_df.loc[chart_df['LABEL'] == 'Chloride (serum)',        'LABEL'] = 'Chloride'
chart_df.loc[chart_df['LABEL'] == 'Chloride (100-112)',      'LABEL'] = 'Chloride'
chart_df.loc[chart_df['LABEL'] == 'Chloride  (100-112)',     'LABEL'] = 'Chloride'
chart_df.loc[chart_df['LABEL'] == 'ABG Chloride',            'LABEL'] = 'Chloride'
chart_df.loc[chart_df['LABEL'] == 'Chloride (whole blood)',  'LABEL'] = 'Chloride'

chart_df.loc[chart_df['LABEL'] == 'Potassium (serum)',       'LABEL'] = 'Potassium'
chart_df.loc[chart_df['LABEL'] == 'Potassium (3.5-5.3)',     'LABEL'] = 'Potassium'
chart_df.loc[chart_df['LABEL'] == 'Potassium  (3.5-5.3)',    'LABEL'] = 'Potassium'
chart_df.loc[chart_df['LABEL'] == 'ABG Potassium',           'LABEL'] = 'Potassium'
chart_df.loc[chart_df['LABEL'] == 'ABG POTASSIUM',           'LABEL'] = 'Potassium'
chart_df.loc[chart_df['LABEL'] == 'Potassium (whole blood)', 'LABEL'] = 'Potassium'

chart_df.loc[chart_df['LABEL'] == 'Ionized calcium', 'LABEL'] = 'Ionized Calcium'
chart_df.loc[chart_df['LABEL'] == 'ionized calcium', 'LABEL'] = 'Ionized Calcium'
chart_df.loc[chart_df['LABEL'] == 'IONIZED CALCIUM', 'LABEL'] = 'Ionized Calcium'

chart_df.loc[chart_df['LABEL'] == 'Alkaline Phosphate', 'LABEL'] = 'Alkaline Phosphatase'
chart_df.loc[chart_df['LABEL'] == 'Alk. Phosphate',     'LABEL'] = 'Alkaline Phosphatase'

chart_df.loc[chart_df['LABEL'] == 'bun',           'LABEL'] = 'BUN'
chart_df.loc[chart_df['LABEL'] == 'BUN (6-20)',    'LABEL'] = 'BUN'
chart_df.loc[chart_df['LABEL'] == 'BUN    (6-20)', 'LABEL'] = 'BUN'

chart_df.loc[chart_df['LABEL'] == 'Anion gap',          'LABEL'] = 'Anion Gap' 
chart_df.loc[chart_df['LABEL'] == 'Anion Gap   (8-20)', 'LABEL'] = 'Anion Gap'

chart_df.loc[chart_df['LABEL'] == 'Lactic Acid(0.5-2.0)', 'LABEL'] = 'Lactate'
chart_df.loc[chart_df['LABEL'] == 'Lactic Acid',          'LABEL'] = 'Lactate'

chart_df.loc[chart_df['LABEL'] == 'HCO3',         'LABEL'] = 'Bicarbonate' 
chart_df.loc[chart_df['LABEL'] == 'HCO3 (serum)', 'LABEL'] = 'Bicarbonate'

chart_df.loc[chart_df['LABEL'] == 'Creatinine (0-1.3)',   'LABEL'] = 'Creatinine'
chart_df.loc[chart_df['LABEL'] == 'Creatinine   (0-0.7)', 'LABEL'] = 'Creatinine'

chart_df.loc[chart_df['LABEL'] == 'Hematocrit (serum)', 'LABEL'] = 'Hematocrit'
chart_df.loc[chart_df['LABEL'] == 'Hematocrit (35-51)', 'LABEL'] = 'Hematocrit'

chart_df.loc[chart_df['LABEL'] == 'Bilirubin',            'LABEL'] = 'Bilirubin, Direct'
chart_df.loc[chart_df['LABEL'] == 'Direct Bilirubin',     'LABEL'] = 'Bilirubin, Direct'
chart_df.loc[chart_df['LABEL'] == 'Direct Bili',          'LABEL'] = 'Bilirubin, Direct'
chart_df.loc[chart_df['LABEL'] == 'Direct Bili (0-0.3)',  'LABEL'] = 'Bilirubin, Direct'
chart_df.loc[chart_df['LABEL'] == 'Indirect Bili(0-1.0)', 'LABEL'] = 'Bilirubin, Indirect'
chart_df.loc[chart_df['LABEL'] == 'Total Bilirubin',      'LABEL'] = 'Bilirubin, Total'
chart_df.loc[chart_df['LABEL'] == 'Total Bili (0-1.5)',   'LABEL'] = 'Bilirubin, Total'
chart_df.loc[chart_df['LABEL'] == 'Total Bili',           'LABEL'] = 'Bilirubin, Total'

chart_df.loc[chart_df['LABEL'] == 'Arterial Base Excess', 'LABEL'] = 'Base Excess'
chart_df.loc[chart_df['LABEL'] == 'Venous Base Excess',   'LABEL'] = 'Base Excess'
chart_df.loc[chart_df['LABEL'] == 'Base Excess (cap)',    'LABEL'] = 'Base Excess' 

chart_df.loc[chart_df['LABEL'] == 'PH',            'LABEL'] = 'pH'
chart_df.loc[chart_df['LABEL'] == 'Ph',            'LABEL'] = 'pH'
chart_df.loc[chart_df['LABEL'] == 'ph level',      'LABEL'] = 'pH'
chart_df.loc[chart_df['LABEL'] == 'PH (Arterial)', 'LABEL'] = 'pH'
chart_df.loc[chart_df['LABEL'] == 'pH (Art)',      'LABEL'] = 'pH'
chart_df.loc[chart_df['LABEL'] == 'Arterial pH',   'LABEL'] = 'pH'
chart_df.loc[chart_df['LABEL'] == 'Art.pH',        'LABEL'] = 'pH'
chart_df.loc[chart_df['LABEL'] == 'PH (Venous)',   'LABEL'] = 'pH'
chart_df.loc[chart_df['LABEL'] == 'Venous pH',     'LABEL'] = 'pH'
chart_df.loc[chart_df['LABEL'] == 'pH (cap)',      'LABEL'] = 'pH'

chart_df.loc[chart_df['LABEL'] == 'FIBRINOGEN',           'LABEL'] = 'Fibrinogen'
chart_df.loc[chart_df['LABEL'] == 'Fibrinogen (150-400)', 'LABEL'] = 'Fibrinogen'

chart_df.loc[chart_df['LABEL'] == 'LDH', 'LABEL'] = 'Lactate Dehydrogenase (LD)'

chart_df.loc[chart_df['LABEL'] == 'Troponin-T', 'LABEL'] = 'Troponin T'
chart_df.loc[chart_df['LABEL'] == 'Troponin',   'LABEL'] = 'Troponin I'

chart_df.loc[chart_df['LABEL'] == 'Triglyceride',         'LABEL'] = 'Triglycerides'
chart_df.loc[chart_df['LABEL'] == 'Triglyceride (0-200)', 'LABEL'] = 'Triglycerides' 
chart_df.loc[chart_df['LABEL'] == 'Triglyceride (0-250)', 'LABEL'] = 'Triglycerides'

chart_df.loc[chart_df['LABEL'] == 'Vancomycin/Trough', 'LABEL'] = 'Vancomycin (Trough)'
chart_df.loc[chart_df['LABEL'] == 'Vancomycin/Peak',   'LABEL'] = 'Vancomycin (Peak)'
chart_df.loc[chart_df['LABEL'] == 'Vancomycin/Random', 'LABEL'] = 'Vancomycin (Random)'

chart_df.loc[chart_df['LABEL'] == 'PEEP Set', 'LABEL'] = 'PEEP (Set)'
chart_df.loc[chart_df['LABEL'] == 'PEEP set', 'LABEL'] = 'PEEP (Set)'
chart_df.loc[chart_df['LABEL'] == 'MEASURED PEEP',    'LABEL'] = 'PEEP'
chart_df.loc[chart_df['LABEL'] == 'Intrinsic peep',   'LABEL'] = 'PEEP'
chart_df.loc[chart_df['LABEL'] == 'Total PEEP Level', 'LABEL'] = 'PEEP'
chart_df.loc[chart_df['LABEL'] == 'total PeeP',       'LABEL'] = 'PEEP'
chart_df.loc[chart_df['LABEL'] == 'Auto-PEEP level',  'LABEL'] = 'PEEP'
chart_df.loc[chart_df['LABEL'] == 'Auto-PEEP Level',  'LABEL'] = 'PEEP'
chart_df.loc[chart_df['LABEL'] == 'autopeep',         'LABEL'] = 'PEEP'
chart_df.loc[chart_df['LABEL'] == 'Autopeep',         'LABEL'] = 'PEEP'
chart_df.loc[chart_df['LABEL'] == 'AUTOPeeP',         'LABEL'] = 'PEEP'
chart_df.loc[chart_df['LABEL'] == 'auto-peep',        'LABEL'] = 'PEEP'
chart_df.loc[chart_df['LABEL'] == 'Auto PeeP',        'LABEL'] = 'PEEP'

chart_df.loc[chart_df['LABEL'] == 'Tidal Volume (set)', 'LABEL'] = 'Tidal Volume (Set)'
chart_df.loc[chart_df['LABEL'] == 'tidal volumes',              'LABEL'] = 'Tidal Volume'
chart_df.loc[chart_df['LABEL'] == 'TIDAL VOLUME',               'LABEL'] = 'Tidal Volume'
chart_df.loc[chart_df['LABEL'] == 'tidal volume',               'LABEL'] = 'Tidal Volume' 
chart_df.loc[chart_df['LABEL'] == 'tidal vol',                  'LABEL'] = 'Tidal Volume'
chart_df.loc[chart_df['LABEL'] == 'Tidal Volume (Obser)',       'LABEL'] = 'Tidal Volume'
chart_df.loc[chart_df['LABEL'] == 'Tidal Volume (observed)',    'LABEL'] = 'Tidal Volume'
chart_df.loc[chart_df['LABEL'] == 'Tidal Volume (spontaneous)', 'LABEL'] = 'Tidal Volume'
chart_df.loc[chart_df['LABEL'] == 'spont tidal volumes',        'LABEL'] = 'Tidal Volume'
chart_df.loc[chart_df['LABEL'] == 'Spont. Tidal Volume',        'LABEL'] = 'Tidal Volume'
chart_df.loc[chart_df['LABEL'] == 'Tidal Volume (Spont)',       'LABEL'] = 'Tidal Volume'
chart_df.loc[chart_df['LABEL'] == 'spont Tidal volumes',        'LABEL'] = 'Tidal Volume'
chart_df.loc[chart_df['LABEL'] == 'SPNIOT TIDAL VOLUMES',       'LABEL'] = 'Tidal Volume'
chart_df.loc[chart_df['LABEL'] == 'sp tidal volumes',           'LABEL'] = 'Tidal Volume'

chart_df.loc[chart_df['LABEL'] == 'lipase', 'LABEL'] = 'Lipase'

chart_df.loc[chart_df['LABEL'] == 'Total Protein(6.5-8)', 'LABEL'] = 'Total Protein'
chart_df.loc[chart_df['LABEL'] == 'T. Protein (5-7.5)',   'LABEL'] = 'Total Protein'

chart_df.loc[chart_df['LABEL'] == 'ammonia',              'LABEL'] = 'Ammonia' 
chart_df.loc[chart_df['LABEL'] == 'AMMONIA',              'LABEL'] = 'Ammonia'
chart_df.loc[chart_df['LABEL'] == 'AMMONIA/12-47 UMOL/L', 'LABEL'] = 'Ammonia'

chart_df.loc[chart_df['LABEL'] == 'RBC(3.6-6.2)', 'LABEL'] = 'RBC'
chart_df.loc[chart_df['LABEL'] == 'PRBCs',        'LABEL'] = 'RBC'
chart_df.loc[chart_df['LABEL'] == 'PRBCS',        'LABEL'] = 'RBC'
chart_df.loc[chart_df['LABEL'] == 'PRBC',         'LABEL'] = 'RBC'

chart_df.loc[chart_df['LABEL'] == 'cortisol',           'LABEL'] = 'Cortisol'
chart_df.loc[chart_df['LABEL'] == 'CORTISOL LEVEL PRE', 'LABEL'] = 'Cortisol' 

chart_df.loc[chart_df['LABEL'] == 'ALBUMIN',            'LABEL'] = 'Albumin'
chart_df.loc[chart_df['LABEL'] == 'albumin',            'LABEL'] = 'Albumin'
chart_df.loc[chart_df['LABEL'] == 'Albumin (>3.2)',     'LABEL'] = 'Albumin'
chart_df.loc[chart_df['LABEL'] == 'Albumin  (3.9-4.8)', 'LABEL'] = 'Albumin'

chart_df.loc[chart_df['LABEL'] == 'pressure support', 'LABEL'] = 'Pressure Support'
chart_df.loc[chart_df['LABEL'] == 'PRESSURE SUPPORT', 'LABEL'] = 'Pressure Support'

chart_df.loc[chart_df['LABEL'] == 'Arterial CO2(Calc)',   'LABEL'] = 'Total CO2'
chart_df.loc[chart_df['LABEL'] == 'CaO2',                 'LABEL'] = 'Total CO2'
chart_df.loc[chart_df['LABEL'] == 'Venous CO2',           'LABEL'] = 'Total CO2'
chart_df.loc[chart_df['LABEL'] == 'Carbon Dioxide',       'LABEL'] = 'Total CO2'
chart_df.loc[chart_df['LABEL'] == 'Venous CO2(Calc)',     'LABEL'] = 'Total CO2'
chart_df.loc[chart_df['LABEL'] == 'TCO2 (calc) Arterial', 'LABEL'] = 'Total CO2'
chart_df.loc[chart_df['LABEL'] == 'TCO2 (calc) Venous',   'LABEL'] = 'Total CO2'
chart_df.loc[chart_df['LABEL'] == 'TCO2 (cap)',           'LABEL'] = 'Total CO2'
chart_df.loc[chart_df['LABEL'] == 'TcCO2 [Value]',        'LABEL'] = 'Total CO2'
chart_df.loc[chart_df['LABEL'] == 'TcO2 [Value]',         'LABEL'] = 'Total CO2'
chart_df.loc[chart_df['LABEL'] == 'TCO2 (calc) Venous',   'LABEL'] = 'Total CO2'
chart_df.loc[chart_df['LABEL'] == 'Venous TCO2',          'LABEL'] = 'Total CO2'
chart_df.loc[chart_df['LABEL'] == 'TCO2        (21-30)',  'LABEL'] = 'Total CO2'

chart_df.loc[chart_df['LABEL'] == 'C Reactive Protein (CRP)', 'LABEL'] = 'C-Reactive Protein (CRP)'

In [23]:
chart_df.head(2)

### Prescriptions

In [24]:
prescriptions = pd.read_csv(path_csv + "prescriptions.csv")
prescriptions.head(2)

### Date&Time events

In [25]:
datetimeevents = pd.read_csv(path_csv  + "datetimeevents.csv")
datetimeevents.head(2)

### Output events

In [26]:
outputevents = pd.read_csv(path_csv  + "outputevents.csv")
outputevents.head(2)

### Input events

In [27]:
inputevents = pd.read_csv(path_csv + "inputevents.csv")
inputevents.head(2)

### Blood culture

In [28]:
culture_df = pd.read_csv(path_csv + "culture_event.csv")
culture_df.head(2)

### Microbiology test

In [29]:
microbio_df = pd.read_csv(path_csv + "microbio_event.csv")
microbio_df.head(2)

### Notes 

In [30]:
note = pd.read_csv(path_csv + "note.csv")
note.head(2)

### Discharge notes

In [31]:
discharge_note = pd.read_csv(path_csv + "discharge_note.csv")
discharge_note.head(2)

### All Tables

In [32]:
tables = [lab_df, chart_df, prescriptions, datetimeevents, outputevents, inputevents, 
          culture_df, microbio_df, note, discharge_note]

In [33]:
all_tables = pd.concat(tables)
all_tables[['ICUSTAY_ID']] = all_tables[['ICUSTAY_ID']].astype(int)
all_tables = all_tables.sort_values(by=['ICUSTAY_ID', 'CHARTTIME'], axis=0)
all_tables.reset_index(inplace=True, drop=True)

In [34]:
all_tables.head(2)

### Save Data 

In [35]:
def cohort_stay_id(frame):
    cohort = frame.ICUSTAY_ID.unique()
    return cohort

### ICU Admission Information

In [36]:
def break_up_admission_by_unit_stay(admission, output_path, stayid, verbose=1):
    
    unit_stays = stayid
    nb_unit_stays = unit_stays.shape[0]
    
    for i, icu_stay_id in enumerate(unit_stays):
        
        if verbose:
            sys.stdout.write('\rStayID {0} of {1}...'.format(i+1, nb_unit_stays))
            
        dn = os.path.join(output_path, str(icu_stay_id))
        
        try:
            os.makedirs(dn)  
        except:
            pass

        admission.loc[admission.ICUSTAY_ID == icu_stay_id].to_csv(os.path.join(dn, 'admission.csv'), index=False)
    
    if verbose:
        sys.stdout.write('DONE!\n')

In [ ]:
icu_stayid_adm  = cohort_stay_id(cohort_df)
break_up_admission_by_unit_stay(cohort_df, path_timeseries, icu_stayid_adm, verbose=1)

### ICU patients comorbidities

In [38]:
def break_up_comorbidities_by_unit_stay(comorbidity, output_path, stayid, verbose=1):
    
    unit_stays = stayid
    nb_unit_stays = unit_stays.shape[0]
    
    for i, icu_stay_id in enumerate(unit_stays):
        
        if verbose:
            sys.stdout.write('\rStayID {0} of {1}...'.format(i+1, nb_unit_stays))
            
        dn = os.path.join(output_path, str(icu_stay_id))
        
        try:
            os.makedirs(dn)  
        except:
            pass

        comorbidity.loc[comorbidity.ICUSTAY_ID == icu_stay_id].to_csv(os.path.join(dn, 'comorbidity.csv'), index=False)
    
    if verbose:
        sys.stdout.write('DONE!\n')

In [39]:
icu_stayid_comorbidity  = cohort_stay_id(comorbidities)
break_up_comorbidities_by_unit_stay(comorbidities, path_timeseries, icu_stayid_comorbidity, verbose=1)

StayID 19351 of 19351...DONE!


### All Tables

In [40]:
def break_up_all_tables_by_unit_stay(all_tables, output_path, stayid, verbose=1):
    
    unit_stays = stayid
    nb_unit_stays = unit_stays.shape[0]
    
    for i, icu_stay_id in enumerate(unit_stays):
        
        if verbose:
            sys.stdout.write('\rStayID {0} of {1}...'.format(i+1, nb_unit_stays))
            
        dn = os.path.join(output_path, str(icu_stay_id))
        
        try:
            os.makedirs(dn)  
        except:
            pass

        all_tables.loc[all_tables.ICUSTAY_ID == icu_stay_id].to_csv(os.path.join(dn, 'all_tables.csv'), index=False)
    
    if verbose:
        sys.stdout.write('DONE!\n')

In [41]:
stay_id_all_tables  = cohort_stay_id(all_tables)
break_up_all_tables_by_unit_stay(all_tables, path_timeseries, stay_id_all_tables, verbose=1)

StayID 19341 of 19341...DONE!


### Save list of unique variables

In [42]:
all_variables = [

'Microbio Test', 'Blood Culture', 'Note', 'Discharge_Note', 
 
'22 Gauge Insertion Date', '20 Gauge Insertion Date', '18 Gauge Insertion Date', 'Arterial line Insertion Date',
'Arterial line Tubing Change', 'Arterial Line Dressing Change', 'Multi Lumen Insertion Date',
'Multi Lumen Cap Change', 'Multi Lumen Dressing Change', 'Multi Lumen Tubing Change',
    
'Antibiotic_PRC', 'Fentanyl_PRC', 'Propofol_PRC', 'Norepinephrine_PRC', 'Insulin_PRC', 'Midazolam_PRC',
'Heparin_PRC', 'Dexmedetomidine_PRC', 'Amiodarone_PRC', 'Vasopressin_PRC', 'Phenylephrine_PRC', 'Dopamine_PRC',
'Nicardipine_PRC', 'Milrinone_PRC', 'Pantoprazole_PRC', 'Diltiazem_PRC', 'Dobutamine_PRC', 'Nitroglycerin_PRC',
'Epinephrine_PRC', 'Warfarin_PRC', 'Apixaban_PRC', 'Dabigatran_PRC', 'Rivaroxaban_PRC', 'Edoxaban_PRC',

'UrineOutput_IO', 'Stool_IO', 'Propofol_IO', 'Fentanyl_IO', 'Insulin_IO', 'Heparin_IO', 'Midazolam_IO',
'Dexmedetomidine_IO', 'Vassopressin_IO', 'Albumin_IO', 'Ceftriaxone_IO', 'Cefazolin_IO', 'Cefepime_IO',
'Ceftazidime_IO', 'Vancomycin_IO', 'Clindamycin_IO', 'Metronidazole_IO', 'Meropenem_IO', 'Acyclovir_IO', 
'Azithromycin_IO', 'Levofloxacin_IO', 'Micafungin_IO', 'Fluconazole_IO', 'Thiamine_IO', 'Dobutamine_IO', 
'Milrinone_IO', 'Fluids_IO', 'OralIntake_IO', 'P.O._IO', 'SodiumChloride_IO', 'IVPB_IO', 'Crystalloids_IO',
'NSIVF_IO', 'Norepinephrine_IO', 'Amiodarone_IO', 'Phenylephrine_IO', 'Epinephrine_IO', 'Nicardipine_IO',
'Pantoprazole_IO', 'Diltiazem_IO', 'Nitroglycerin_IO', 'NeuroblockAgent_IO', 'K-IV_IO', 'Ca-IV_IO', 'Ca-nonIV_IO',
'Mg-IV_IO', 'Mg-nonIV_IO', 'P-IV_IO', 'P-nonIV_IO', 'BetaBlockers_IO', 'CaBlockers_IO', 'LoopDiuretics_IO',
'TPNutrition_IO', 'PNutrition_IO', 'Dextrose_IO', 'POnutrition_IO', 'Vasopressors_IO',

'Temperature', 'PT', 'PTT', 'INR(PT)', 'pH', 'Lactate', 'Lactate Dehydrogenase (LD)', 'Base Excess', 
'Anion Gap', 'Bicarbonate', 'Creatinine', 'Hematocrit', 'Hemoglobin',  'Bilirubin, Total', 'Bilirubin, Direct', 
'Bilirubin, Indirect', 'BUN', 'MCV', 'MCH', 'MCHC', 'RDW', 'RBC', 'WBC', 'Red Blood Cells', 'White Blood Cells', 
'Platelet Count', 'Glucose', 'Ammonia', 'Magnesium', 'Phosphate', 'Alkaline Phosphatase', 'Potassium', 'Sodium',  
'Chloride', 'Ionized Calcium', 'Calcium, Total', 'Cholesterol, Total', 'Cholesterol, HDL', 'Cholesterol, LDL',
'C-Reactive Protein (CRP)', 'pO2', 'pCO2', 'ALT', 'AST', 'Amylase',  'Lipase', 
'Absolute Lymphocyte Count', 'Monocyte Count', 'Eosinophil Count', 'O2 Flow', 'Oxygen', 'Oxygen Saturation', 
'Total CO2', 'Albumin', 'Troponin T', 'Troponin I', 'Vancomycin', 'Triglycerides', 'Fibrinogen', 'Transferrin', 
'Ferritin', 'Cortisol', 'Protein', 'Total Protein', 'PEEP', 'Tidal Volume', 'Intubated', 'Ventilator', 
'Ventilation Rate', 'Granulocyte Count', 'Promyelocytes', 'Metamyelocytes', 'Myelocytes', 'Ovalocytes', 

'Heart Rhythm', 'Heart Rate', 'Skin Temperature',  
'Respiratory Rate', 'Respiratory Rate (Total)', 'Respiratory Rate (Set)', 'Non Invasive Blood Pressure mean', 
'Non Invasive Blood Pressure diastolic', 'Non Invasive Blood Pressure systolic' , 'Arterial Blood Pressure mean',
'Arterial Blood Pressure diastolic', 'Arterial Blood Pressure systolic',  'Pulmonary Artery Pressure mean',
'Pulmonary Artery Pressure diastolic',  'Pulmonary Artery Pressure systolic', 'Mean Airway Pressure',
    
'Pain Level', 'Pain Present', 'GCS Total', 'GCS - Eye Opening',  'GCS - Motor Response' , 'GCS - Verbal Response', 
'Mental status', 'Richmond-RAS Scale', 'Goal Richmond-RAS Scale', 'Risk for Falls', 'Delirium assessment',
'CAM-ICU MS Change', 'CAM-ICU RASS LOC', 'CAM-ICU Inattention', 'CAM-ICU Altered LOC',
'CAM-ICU Disorganized thinking', 'Sedation Score', 'Sedation Scale',  'EtCO2 Clinical indication',
    
'Flow Rate (L/min)', 'FiO2', 'FiO2 (Set)', 'SpO2', 'SvO2',  'Admission Weight (Kg)', 'Height (cm)', 'CVP', 
'ETCO2',  'Calcium non-ionized', 'Plateau Pressure', 'Vancomycin (Trough)', 'Vancomycin (Peak)', 
'Vancomycin (Random)', 'PEEP (Set)', 'Tidal Volume (Set)', 'Ventilator Mode', 'Ventilator Type', 
'Differential-Eos', 'Differential-Basos', 'Differential-Lymphs', 'Differential-Neuts', 'Differential-Monos', 
'Differential-Bands', 'Differential-Polys', 'Pressure Support',

]

cat_int_value = ['Intubated', 'Skin Temperature', 'Pain Level', 'Pain Present', 'GCS Total', 'GCS - Eye Opening', 
                 'GCS - Motor Response' , 'GCS - Verbal Response', 'Mental status', 'Richmond-RAS Scale', 
                 'Goal Richmond-RAS Scale', 'Risk for Falls', 'Delirium assessment', 'CAM-ICU MS Change', 
                 'CAM-ICU RASS LOC', 'CAM-ICU Inattention', 'CAM-ICU Altered LOC', 'CAM-ICU Disorganized thinking',
                 'Microbio Test', 'Blood Culture']

cat_text_value = ['Heart Rhythm', 'Ventilator Mode', 'Ventilator Type', 'Ventilator',
                  'Sedation Score', 'Sedation Scale', 'EtCO2 Clinical indication']

In [43]:
with open("../Data/csvExtract/variables.txt", "w") as f:
    for variable in all_variables:
        f.write(variable +"\n")